In [ ]:
!pip install sacrebleu evaluate sacremoses bert-score rouge_score openai
!pip install --upgrade transformers


In [18]:
import os
import torch
from datasets import load_dataset
import evaluate
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
import time
from openai import OpenAI

# --- 1. Load Evaluation Metrics ---
bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")


# --- 2. Generalized Helper Functions ---

def sample_data(dataset, n=50):
    """Randomly samples n data points from the dataset."""
    if n > len(dataset):
        n = len(dataset)
    return dataset.shuffle(seed=42).select(range(n))

def llm_inference(api_call_func, inputs, batch_size=4):
    """
    Performs inference for LLMs via API calls, processing in batches.
    """
    results = []
    for i in tqdm(range(0, len(inputs), batch_size), desc=f"LLM Inference for {api_call_func.__name__}"):
        batch = inputs[i:i+batch_size]
        batch_results = api_call_func(batch)
        if not batch_results:
            print(f"[WARNING] No results for batch {i//batch_size}")
        results.extend(batch_results)
    return results

def calculate_metrics_translation(predictions, references):
    """
    Calculates and returns a dictionary of metrics for translation.
    """
    results = {}
    # BLEU and METEOR for translation
    results['BLEU'] = bleu.compute(predictions=predictions, references=[[ref] for ref in references])["score"]
    results['METEOR'] = meteor.compute(predictions=predictions, references=references)["meteor"]
    return results

def evaluate_translation_task(models, dataset, task_config):
    """
    Main function to orchestrate the evaluation process for the translation task.
    """
    # Correctly extract nested 'en' and 'fr' fields
    inputs = [item[task_config["input_key"]]['en'] for item in dataset]
    references_all = [item[task_config["ref_key"]]['fr'] for item in dataset]

    results_df = pd.DataFrame()

    for name, api_func in models.items():
        print(f"\n[EVALUATING] Model: {name}")

        preds = llm_inference(api_func, inputs)

        # Filter out any empty or invalid prediction/reference pairs
        valid_pairs = [(p, r) for p, r in zip(preds, references_all) if p and r]
        if not valid_pairs:
            print(f"[WARNING] Skipping model {name} — no valid prediction/reference pairs found.")
            continue

        valid_preds, valid_refs = zip(*valid_pairs)

        # Compute metrics
        metrics = calculate_metrics_translation(list(valid_preds), list(valid_refs))
        metrics["Model"] = name
        results_df = pd.concat([results_df, pd.DataFrame([metrics])], ignore_index=True)

    return results_df


# --- 3. LLM API Call Functions for Translation ---

translation_system_prompt = "You are an expert translator. Translate the following English text to French. Respond with ONLY the French translation and nothing else."

def safe_deepseek_translation(batch):
    """Handles API calls for the DeepSeek model."""
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=","
    )
    responses = []
    for text in batch:
        try:
            response = client.chat.completions.create(
                model="deepseek/deepseek-coder",
                messages=[
                    {"role": "system", "content": translation_system_prompt},
                    {"role": "user", "content": text}
                ]
            )
            content = response.choices[0].message.content.strip()
            responses.append(content)
        except Exception as e:
            print(f"[ERROR] API call failed for DeepSeek with error: {e}")
            responses.append("")
            time.sleep(1)
    return responses

def safe_mistral_translation(batch):
    """Handles API calls for the Mistral model."""
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=","
    )
    responses = []
    for text in batch:
        try:
            response = client.chat.completions.create(
                model="mistralai/mistral-7b-instruct",
                messages=[
                    {"role": "system", "content": translation_system_prompt},
                    {"role": "user", "content": text}
                ]
            )
            content = response.choices[0].message.content.strip()
            responses.append(content)
        except Exception as e:
            print(f"[ERROR] API call failed for Mistral with error: {e}")
            responses.append("")
            time.sleep(1)
    return responses

def safe_qwen_translation(batch):
    """Handles API calls for the Qwen model."""
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key="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"
    )
    responses = []
    for text in batch:
        try:
            response = client.chat.completions.create(
                model="qwen/qwen3-235b-a22b:free",
                messages=[
                    {"role": "system", "content": translation_system_prompt},
                    {"role": "user", "content": text}
                ]
            )
            content = response.choices[0].message.content.strip()
            responses.append(content)
        except Exception as e:
            print(f"[ERROR] API call failed for Qwen with error: {e}")
            responses.append("")
            time.sleep(1)
    return responses

# --- 4. Main Evaluation Execution ---

# Define the task configuration for translation
TASK_CONFIGS = {
    "translation": {
        "input_key": "translation",
        "ref_key": "translation",
    }
}

# Map model names to their API functions
translation_llms = {
    #"DeepSeek": safe_deepseek_translation,
    #"Mistral": safe_mistral_translation,
    "Qwen": safe_qwen_translation,
}

# Load and sample the dataset
full_dataset = load_dataset("wmt14", "fr-en", split="test")
test_set_translation = sample_data(full_dataset, n=50) # Using 50 samples for evaluation

# Evaluate the LLMs
llm_results_df = evaluate_translation_task(translation_llms, test_set_translation, TASK_CONFIGS["translation"])

# Display results
print("\n--- LLM Translation Evaluation Results ---")
print(llm_results_df.to_string())

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]


[EVALUATING] Model: Qwen


LLM Inference for safe_qwen_translation:  85%|████████▍ | 11/13 [23:54<03:35, 107.62s/it]

[ERROR] API call failed for Qwen with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753920000000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for Qwen with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753878060000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for Qwen with error: Error code: 429 - {'error': {'message': 'Rate limit exceed

LLM Inference for safe_qwen_translation:  92%|█████████▏| 12/13 [24:18<01:22, 82.07s/it] 

[ERROR] API call failed for Qwen with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753920000000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for Qwen with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753878120000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}


LLM Inference for safe_qwen_translation: 100%|██████████| 13/13 [24:24<00:00, 112.67s/it]



--- LLM Translation Evaluation Results ---
       BLEU    METEOR Model
0  3.356019  0.499626  Qwen


In [16]:
import os
import torch
from datasets import load_dataset
import evaluate
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
import time
from openai import OpenAI
from sklearn.metrics import accuracy_score
import re
# --- 2. Generalized Helper Functions ---
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")
def sample_data(dataset, n=50):
    """Randomly samples n data points from the dataset."""
    if n > len(dataset):
        n = len(dataset)
    return dataset.shuffle(seed=42).select(range(n))

def llm_inference(api_call_func, inputs, batch_size=4):
    """
    Performs inference for LLMs via API calls, processing in batches.
    """
    results = []
    for i in tqdm(range(0, len(inputs), batch_size), desc=f"LLM Inference for {api_call_func.__name__}"):
        batch = inputs[i:i+batch_size]
        batch_results = api_call_func(batch)
        results.extend(batch_results)
    return results

def extract_final_answer(text):
    """
    Extracts the final answer from the model's full CoT response.
    It looks for phrases like "The answer is" or the last numerical value.
    """
    # Look for "The answer is X" patterns, case-insensitive
    match = re.search(r'(?i)(?:the final answer is|the answer is|is:)\s*(-?\d+\.?\d*)\b', text)
    if match:
        return match.group(1)

    # If no explicit answer phrase, find all numbers and return the last one
    numbers = re.findall(r'-?\d+\.?\d*', text)
    if numbers:
        return numbers[-1]
    return "" # Return empty if no answer is found

def calculate_cot_metrics(predictions, references):
    """
    Calculates metrics for the Chain-of-Thought task.
    - Answering Accuracy: Compares the extracted final answer.
    - Reasoning Accuracy: Compares the full rationale using ROUGE and BERTScore.
    """
    results = {}
    
    # Extract rationales and answers from references
    ref_answers = [ref['answer'] for ref in references]
    ref_rationales = [ref['rationale'] for ref in references]

    # Extract final answers from predictions
    pred_answers = [extract_final_answer(p) for p in predictions]

    # 1. Answering Accuracy
    correct_answers = 0
    for pred, ref in zip(pred_answers, ref_answers):
        # Normalize by removing punctuation/whitespace for a more robust comparison
        if pred.strip().strip('.') == ref.strip().strip('.'):
            correct_answers += 1
    
    results['Answering Accuracy'] = correct_answers / len(ref_answers) if ref_answers else 0

    # 2. Reasoning Accuracy (using ROUGE and BERTScore on the rationale)
    results['Reasoning ROUGE-L'] = rouge.compute(predictions=predictions, references=ref_rationales)["rougeL"]
    
    bert_results = bertscore.compute(predictions=predictions, references=ref_rationales, lang="en")
    results['Reasoning BERTScore_F1'] = np.mean(bert_results['f1'])

    return results

def evaluate_cot_task(models, dataset, task_config):
    """
    Main function to orchestrate the CoT evaluation process.
    """
    inputs = [item[task_config["input_key"]] for item in dataset]
    # Create a list of dictionaries for references
    references_all = [{"rationale": item[task_config["ref_rationale"]], "answer": item[task_config["ref_answer"]]} for item in dataset]

    results_df = pd.DataFrame()

    for name, api_func in models.items():
        print(f"\n[EVALUATING] Model: {name}")

        preds = llm_inference(api_func, inputs)

        # Compute metrics
        metrics = calculate_cot_metrics(preds, references_all)
        metrics["Model"] = name
        results_df = pd.concat([results_df, pd.DataFrame([metrics])], ignore_index=True)

    return results_df


# --- 3. LLM API Call Functions for CoT Reasoning ---

cot_system_prompt = "Solve the following problem by thinking step-by-step. First, provide your reasoning process, then conclude with the final answer in the format 'The answer is [your answer]'."

def safe_api_call(model_name, api_key, batch):
    """A robust function to handle API calls for a batch of inputs."""
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    responses = []
    for text in batch:
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": cot_system_prompt},
                    {"role": "user", "content": text}
                ]
            )
            content = response.choices[0].message.content.strip()
            responses.append(content)
        except Exception as e:
            print(f"[ERROR] API call failed for model {model_name} with error: {e}")
            responses.append("") # Append an empty string on failure
            time.sleep(1)
    return responses

# Create specific functions for each model, each with its own API key
def safe_deepseek_cot(batch):
    api_key = "HERE" 
    return safe_api_call("deepseek/deepseek-r1-0528:free", api_key, batch)

def safe_mistral_cot(batch):
    api_key = "HERE"
    return safe_api_call("mistralai/mistral-7b-instruct", api_key, batch)

def safe_qwen_cot(batch):
    api_key = "HERE"
    return safe_api_call("qwen/qwen3-235b-a22b:free", api_key, batch)


# --- 4. Main Evaluation Execution ---

# Define the task configuration
TASK_CONFIGS = {
    "cot_reasoning": {
        "input_key": "source",
        "ref_rationale": "rationale",
        "ref_answer": "target"
    }
}

# Map model names to their API functions
cot_llms = {
    "DeepSeek": safe_deepseek_cot,
    "Mistral": safe_mistral_cot,
    "Qwen": safe_qwen_cot,
}

# Load and sample the dataset
full_dataset = load_dataset("kaist-ai/CoT-Collection", "en", split="train")
test_set_cot = sample_data(full_dataset, n=50) 

# Evaluate the LLMs
llm_results_df = evaluate_cot_task(cot_llms, test_set_cot, TASK_CONFIGS["cot_reasoning"])

# Display results
print("\n--- LLM Chain-of-Thought Evaluation Results ---")
print(llm_results_df.to_string())


[EVALUATING] Model: Qwen


LLM Inference for safe_qwen_cot:  77%|███████▋  | 10/13 [28:55<08:21, 167.11s/it]

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753873920000'}, 'provider_name': None}}, 'user_id': 'user_30aPFGMDOkcHgr3uYlZYN71lo4I'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753920000000'}, 'provider_name': None}}, 'user_id': 'user_30aPFGMDOkcHgr3uYlZYN71lo4I'}


LLM Inference for safe_qwen_cot:  85%|████████▍ | 11/13 [29:32<04:14, 127.12s/it]

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753920000000'}, 'provider_name': None}}, 'user_id': 'user_30aPFGMDOkcHgr3uYlZYN71lo4I'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753920000000'}, 'provider_name': None}}, 'user_id': 'user_30aPFGMDOkcHgr3uYlZYN71lo4I'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



--- LLM Chain-of-Thought Evaluation Results ---
   Answering Accuracy  Reasoning ROUGE-L  Reasoning BERTScore_F1 Model
0                0.08            0.15871                0.717894  Qwen


In [20]:
import os
import torch
from datasets import load_dataset
import numpy as np
import pandas as pd
import random
from tqdm import tqdm
import time
from openai import OpenAI
from sklearn.metrics import accuracy_score
import re
# --- 1. Generalized Helper Functions ---

def sample_data(dataset, n=50):
    """Filters out invalid labels and randomly samples n data points."""
    # SNLI dataset uses -1 for labels with no consensus
    dataset = dataset.filter(lambda example: example['label'] != -1)
    if n > len(dataset):
        n = len(dataset)
    return dataset.shuffle(seed=42).select(range(n))

def llm_inference(api_call_func, inputs, batch_size=4):
    """
    Performs inference for LLMs via API calls, processing in batches.
    """
    results = []
    for i in tqdm(range(0, len(inputs), batch_size), desc=f"LLM Inference for {api_call_func.__name__}"):
        batch = inputs[i:i+batch_size]
        batch_results = api_call_func(batch)
        if not batch_results:
            print(f"[WARNING] No results for batch {i//batch_size}")
        results.extend(batch_results)
    return results

def calculate_nli_metrics(predictions, references):
    """
    Calculates accuracy for the NLI task.
    """
    results = {}
    results['Accuracy'] = accuracy_score(references, predictions)
    return results

def evaluate_nli_task(models, dataset, task_config):
    """
    Main function to orchestrate the evaluation process for the NLI task.
    """
    # Create prompts by combining premise and hypothesis
    inputs = [
        f"Premise: {item[task_config['premise_key']]}\nHypothesis: {item[task_config['hypothesis_key']]}"
        for item in dataset
    ]
    references = [item[task_config["ref_key"]] for item in dataset]

    results_df = pd.DataFrame()

    for name, api_func in models.items():
        print(f"\n[EVALUATING] Model: {name}")

        preds = llm_inference(api_func, inputs)

        # Compute metrics
        metrics = calculate_nli_metrics(preds, references)
        metrics["Model"] = name
        results_df = pd.concat([results_df, pd.DataFrame([metrics])], ignore_index=True)

    return results_df


# --- 2. LLM API Call Functions for NLI ---

nli_system_prompt = (
    "You are an expert in Natural Language Inference. Your task is to determine the relationship between a premise and a hypothesis. "
    "The relationship can be 'entailment', 'neutral', or 'contradiction'. "
    "Respond with ONLY one of these three words."
)

def parse_nli_response(text_response):
    """Converts the model's text output to a numerical label."""
    text_response = text_response.lower().strip()
    if "entailment" in text_response:
        return 0
    elif "neutral" in text_response:
        return 1
    elif "contradiction" in text_response:
        return 2
    else:
        # If the model's output is not one of the three, return a default
        # that will be marked as incorrect (e.g., -1).
        return -1


def safe_api_call(model_name, api_key, batch):
    """A robust function to handle API calls for a batch of inputs."""
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    responses = []
    for text in batch:
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": nli_system_prompt},
                    {"role": "user", "content": text}
                ]
            )
            content = response.choices[0].message.content.strip()
            # Parse the text response into a numerical label
            label = parse_nli_response(content)
            responses.append(label)
        except Exception as e:
            print(f"[ERROR] API call failed for model {model_name} with error: {e}")
            responses.append(-1) # Append an invalid label on failure
            time.sleep(1)
    return responses

# Create specific functions for each model, each with its own API key
def safe_deepseek_nli(batch):
    api_key = "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"
    return safe_api_call("deepseek/deepseek-r1-0528:free", api_key, batch)

def safe_mistral_nli(batch):
    api_key = "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"
    return safe_api_call("mistralai/mistral-7b-instruct", api_key, batch)

def safe_qwen_nli(batch):
    api_key = "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"
    return safe_api_call("qwen/qwen3-235b-a22b:free", api_key, batch)


# --- 3. Main Evaluation Execution ---

# Define the task configuration
TASK_CONFIGS = {
    "nli": {
        "premise_key": "premise",
        "hypothesis_key": "hypothesis",
        "ref_key": "label",
    }
}

# Map model names to their API functions
nli_llms = {
    #"DeepSeek": safe_deepseek_nli,
    #"Mistral": safe_mistral_nli,
    "Qwen": safe_qwen_nli,
}

# Load and sample the dataset
full_dataset = load_dataset("stanfordnlp/snli", split="test")
test_set_nli = sample_data(full_dataset, n=50) 

# Evaluate the LLMs
llm_results_df = evaluate_nli_task(nli_llms, test_set_nli, TASK_CONFIGS["nli"])

# Display results
print("\n--- LLM NLI Evaluation Results ---")
print(llm_results_df.to_string())


[EVALUATING] Model: Qwen


LLM Inference for safe_qwen_nli:   0%|          | 0/13 [00:00<?, ?it/s]

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753879020000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753879080000'}, 'provider_na

LLM Inference for safe_qwen_nli:   8%|▊         | 1/13 [00:13<02:41, 13.46s/it]

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '20', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753879080000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '20', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753879080000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limit

...

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753879200000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '20', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753879200000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'mess

LLM Inference for safe_qwen_nli:  92%|█████████▏| 12/13 [03:01<00:14, 14.44s/it]

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753920000000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753920000000'}, 'provider_name': None}}, 'user_id': 'user_30arVbt9BweMFtosSjc9rNeaKZm'}


LLM Inference for safe_qwen_nli: 100%|██████████| 13/13 [03:09<00:00, 14.55s/it]


--- LLM NLI Evaluation Results ---
   Accuracy Model
0      0.04  Qwen


In [16]:
import evaluate
from sklearn.metrics import accuracy_score


In [ ]:
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 nli1
qwen/qwen3-235b-a22b:free
deepseek/deepseek-r1-0528:free

In [21]:
# --- 1. Load Evaluation Metrics ---
rouge = evaluate.load("rouge")

# --- 2. Generalized Helper Functions ---

def sample_data(dataset, n=50):
    """Randomly samples n data points from the dataset."""
    if n > len(dataset):
        n = len(dataset)
    return dataset.shuffle(seed=42).select(range(n))

def llm_inference(api_call_func, inputs, batch_size=4):
    """
    Performs inference for LLMs via API calls, processing in batches.
    """
    results = []
    for i in tqdm(range(0, len(inputs), batch_size), desc=f"LLM Inference for {api_call_func.__name__}"):
        batch = inputs[i:i+batch_size]
        batch_results = api_call_func(batch)
        if not batch_results:
            print(f"[WARNING] No results for batch {i//batch_size}")
        results.extend(batch_results)
    return results

def calculate_summarization_metrics(predictions, references):
    """
    Calculates ROUGE scores for the summarization task.
    """
    results = {}
    rouge_scores = rouge.compute(predictions=predictions, references=references)
    results['BLEU'] = bleu.compute(predictions=predictions, references=[[ref] for ref in references])["score"]
    results['ROUGE-1'] = rouge_scores['rouge1']
    results['ROUGE-2'] = rouge_scores['rouge2']
    results['ROUGE-L'] = rouge_scores['rougeL']
    return results

def evaluate_summarization_task(models, dataset, task_config):
    """
    Main function to orchestrate the evaluation process for the summarization task.
    """
    inputs = [item[task_config["input_key"]] for item in dataset]
    references_all = [item[task_config["ref_key"]] for item in dataset]

    results_df = pd.DataFrame()

    for name, api_func in models.items():
        print(f"\n[EVALUATING] Model: {name}")

        preds = llm_inference(api_func, inputs)

        # Filter out any empty or invalid prediction/reference pairs
        valid_pairs = [(p, r) for p, r in zip(preds, references_all) if p and r]
        if not valid_pairs:
            print(f"[WARNING] Skipping model {name} — no valid prediction/reference pairs found.")
            continue

        valid_preds, valid_refs = zip(*valid_pairs)

        # Compute metrics
        metrics = calculate_summarization_metrics(list(valid_preds), list(valid_refs))
        metrics["Model"] = name
        results_df = pd.concat([results_df, pd.DataFrame([metrics])], ignore_index=True)

    return results_df


# --- 3. LLM API Call Functions for Code Summarization ---

summarization_system_prompt = "You are an expert programmer. Your task is to write a concise, high-quality docstring that explains what the following code does. Respond with ONLY the docstring/summary and nothing else."

def safe_api_call(model_name, api_key, batch):
    """A robust function to handle API calls for a batch of inputs."""
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    responses = []
    for text in batch:
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": summarization_system_prompt},
                    {"role": "user", "content": text}
                ]
            )
            content = response.choices[0].message.content.strip()
            responses.append(content)
        except Exception as e:
            print(f"[ERROR] API call failed for model {model_name} with error: {e}")
            responses.append("") # Append an empty string on failure
            time.sleep(1)
    return responses

# Create specific functions for each model, each with its own API key
def safe_deepseek_summarization(batch):
    api_key = "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"
    return safe_api_call("deepseek/deepseek-r1-0528:free", api_key, batch)

def safe_mistral_summarization(batch):
    api_key = "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"
    return safe_api_call("mistralai/mistral-7b-instruct", api_key, batch)

def safe_qwen_summarization(batch):
    api_key = "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"
    return safe_api_call("qwen/qwen3-235b-a22b:free", api_key, batch)


# --- 4. Main Evaluation Execution ---

# Define the task configuration
TASK_CONFIGS = {
    "code_summarization": {
        "input_key": "whole_func_string",
        "ref_key": "func_documentation_string",
    }
}

# Map model names to their API functions
summarization_llms = {
    "DeepSeek": safe_deepseek_summarization,
    #"Mistral": safe_mistral_summarization,
    "Qwen": safe_qwen_summarization,
}

# Load and sample the dataset (using the 'python' subset)
full_dataset = load_dataset("code_search_net", "python", split="test")
test_set_summarization = sample_data(full_dataset, n=50)

# Evaluate the LLMs
llm_results_df = evaluate_summarization_task(summarization_llms, test_set_summarization, TASK_CONFIGS["code_summarization"])

# Display results
print("\n--- LLM Code Summarization Evaluation Results ---")
print(llm_results_df.to_string())


[EVALUATING] Model: DeepSeek


LLM Inference for safe_deepseek_summarization: 100%|██████████| 13/13 [46:06<00:00, 212.83s/it]



[EVALUATING] Model: Qwen


LLM Inference for safe_qwen_summarization:   0%|          | 0/13 [00:00<?, ?it/s]

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753883160000'}, 'provider_name': None}}, 'user_id': 'user_30azBVFeGExDhex3D7GD4Jq4hnk'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753883160000'}, 'provider_na

...

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753883280000'}, 'provider_name': None}}, 'user_id': 'user_30azBVFeGExDhex3D7GD4Jq4hnk'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753920000000'}, 'provider_name': None}}, 'user_id': 'user_30azBVFeGExDhex3D7GD4Jq4hnk'}
[ERROR] API call failed for model qwen/qwen3-235b

LLM Inference for safe_qwen_summarization:  92%|█████████▏| 12/13 [02:39<00:13, 13.29s/it]

[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: limit_rpm/qwen/qwen3-235b-a22b-04-28/89cb5be4-feac-4f3c-8fe5-f67bb1dc7ce2. High demand for qwen/qwen3-235b-a22b:free on OpenRouter - limited to 1 requests per minute. Please retry shortly.', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '1', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753883340000'}, 'provider_name': None}}, 'user_id': 'user_30azBVFeGExDhex3D7GD4Jq4hnk'}
[ERROR] API call failed for model qwen/qwen3-235b-a22b:free with error: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '20', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1753883340000'}, 'provider_name': None}}, 'user_id': 'user_30azBVFeGExDhex3D7GD4Jq4hnk'}


LLM Inference for safe_qwen_summarization: 100%|██████████| 13/13 [02:46<00:00, 12.81s/it]

[WARNING] Skipping model Qwen — no valid prediction/reference pairs found.

--- LLM Code Summarization Evaluation Results ---
        BLEU  ROUGE-1   ROUGE-2   ROUGE-L     Model
0  11.267978  0.30204  0.148506  0.243118  DeepSeek
